# HuBMAP Cell-Type Expression

The NIH Common Fund Human BioMolecular Atlas Program, or HuBMAP, maps cells and molecules in human tissues. Add cell-type information while keeping missing values, group sizes, and API limits visible.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
hubmap = pd.read_csv(DATA_DIR / "hubmap_cell_expression.csv")
hubmap.head()

## Data availability

`LMNA` and `PKP2` were not present in the Cells API expression index when the data were collected. Their values stay missing instead of being changed to zero.

In [ ]:
availability = (
    hubmap.loc[:, ["gene_symbol", "availability"]]
    .drop_duplicates()
    .sort_values("gene_symbol")
)
availability

## API query pattern

The Cells API creates a query handle on its server. The function below shows the first request and reports API errors instead of hiding them.

In [ ]:
HUBMAP_CELLS_URL = "https://cells.api.hubmapconsortium.org/api/"


def create_cell_query_handle(
    input_type: str,
    input_values: list[str],
    timeout_seconds: int = 30,
) -> str:
    """Create a HuBMAP Cells API handle for an explicit cell query."""
    form_data = [("input_type", input_type)]
    form_data.extend(("input_set", value) for value in input_values)
    response = requests.post(
        f"{HUBMAP_CELLS_URL}cell/",
        data=form_data,
        timeout=timeout_seconds,
    )
    response.raise_for_status()
    return response.json()["results"][0]["query_handle"]


# Example, intentionally not executed during routine notebook runs:
# heart_handle = create_cell_query_handle("organ", ["Heart"])
# ventricular_handle = create_cell_query_handle("celltype", ["CL:0002131"])

In [ ]:
available = hubmap[hubmap["availability"] == "available"].copy()
expression_matrix = available.pivot(
    index="cell_type_label",
    columns="gene_symbol",
    values="mean_normalized_expression",
)
expression_matrix

In [ ]:
axis = expression_matrix.plot.bar(
    color=["#3d64b3", "#764c82", "#0078ae"],
    figsize=(10, 5),
)
axis.set_ylabel("Mean normalized expression in teaching extract")
axis.set_xlabel("HuBMAP cell type")
axis.set_title("Cell-type context for indexed candidate genes")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## Interpretation

Heart-muscle-cell patterns add cell-type context for three genes. Small groups and the 500-cell teaching sample limit strong claims. Missing indexed values show a coverage gap, not absent expression.